# 📋 Notebook 07 — Scientific Report Generator

---

## Tujuan

Notebook ini membaca hasil aktual dari training & evaluasi, kemudian **mengisi otomatis** bagian placeholder di laporan ilmiah `reports/baseline_cnn_exp01_report.md`.

**Prasyarat:** Notebook 05 dan 06 harus sudah dijalankan terlebih dahulu.

> **Catatan:** Notebook ini adalah orchestrator ringan. Tidak ada logika bisnis di sini.

In [ ]:
# ============================================================
# CELL 1 — IMPORTS & SETUP
# ============================================================
import sys
import json
import warnings
from pathlib import Path
from datetime import date

import pandas as pd
import yaml

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'baseline.yaml'
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

EXP_NAME    = config['experiment']['name']
REPORTS_DIR = PROJECT_ROOT / config['output']['reports_dir']
FIGURES_DIR = PROJECT_ROOT / config['output']['figures_dir']
REPORT_PATH = PROJECT_ROOT / 'reports' / f'{EXP_NAME}_report.md'

print(f'Experiment  : {EXP_NAME}')
print(f'Report path : {REPORT_PATH}')

In [ ]:
# ============================================================
# CELL 2 — LOAD ACTUAL RESULTS
# ============================================================

# Load training history
history_path = REPORTS_DIR / f'{EXP_NAME}_training_history.csv'
assert history_path.exists(), f'Training history tidak ditemukan: {history_path}'
df_history = pd.read_csv(history_path)

# Load experiment summary
summary_path = REPORTS_DIR / f'{EXP_NAME}_experiment_summary.json'
assert summary_path.exists(), f'Experiment summary tidak ditemukan: {summary_path}'
with open(summary_path) as f:
    train_summary = json.load(f)

# Load evaluation metrics
metrics_path = REPORTS_DIR / f'{EXP_NAME}_evaluation_metrics.json'
assert metrics_path.exists(), f'Evaluation metrics tidak ditemukan: {metrics_path}'
with open(metrics_path) as f:
    eval_metrics = json.load(f)

print('✅ Semua artifact berhasil di-load.')
print(f'   Training epochs : {len(df_history)}')
print(f'   Best epoch      : {train_summary["best_epoch"]}')
print(f'   Eval Macro F1   : {eval_metrics["macro_f1"]}')

In [ ]:
# ============================================================
# CELL 3 — TAMPILKAN RINGKASAN HASIL AKTUAL
# ============================================================

CLASS_NAMES = config['data']['class_names']

# Best epoch metrics
best_epoch  = train_summary['best_epoch']
best_row    = df_history[df_history['epoch'] == best_epoch]
if best_row.empty:
    best_row = df_history.iloc[-1:]
best_row = best_row.iloc[0]

print('\n' + '='*65)
print('  RINGKASAN HASIL — CNN BASELINE EXPERIMENT 01')
print('='*65)
print(f'\n  [TRAINING]')
print(f'  Total Epochs      : {len(df_history)}')
print(f'  Best Epoch        : {best_epoch}')
print(f'  Best Train Loss   : {best_row["train_loss"]:.4f}')
print(f'  Best Val Loss     : {best_row["val_loss"]:.4f}')

print(f'\n  [EVALUATION — Validation Set]')
print(f'  ⭐ Macro F1 Score  : {eval_metrics["macro_f1"]:.4f}')
print(f'  Accuracy          : {eval_metrics["accuracy"]:.4f}')

f1_vals    = {cls: eval_metrics['per_class'][cls]['f1'] for cls in CLASS_NAMES.values()}
best_cls   = max(f1_vals, key=f1_vals.get)
worst_cls  = min(f1_vals, key=f1_vals.get)
print(f'\n  Kelas terkuat  : {best_cls} (F1={f1_vals[best_cls]:.4f})')
print(f'  Kelas terlemah : {worst_cls} (F1={f1_vals[worst_cls]:.4f})')
print('='*65)

In [ ]:
# ============================================================
# CELL 4 — UPDATE LAPORAN DENGAN NILAI AKTUAL
# ============================================================
# Mengisi placeholder dalam laporan Markdown dengan hasil nyata

# Baca laporan
assert REPORT_PATH.exists(), f'Laporan tidak ditemukan: {REPORT_PATH}'
report_text = REPORT_PATH.read_text(encoding='utf-8')

# Buat section Training Results yang diisi nilai aktual
train_section = f"""### 6.1 Training Summary

| Metrik | Nilai |
|---|---|
| Total Epochs Dijalankan | {len(df_history)} |
| Best Epoch | {best_epoch} |
| Early Stopped | {'Ya' if train_summary.get('stopped_early', False) else 'Tidak'} |
| Total Training Time | {train_summary.get('total_time_min', 0):.1f} menit |

### 6.2 Training Metrics (Best Epoch)

| Split | Loss | Accuracy | Macro F1 |
|---|---|---|---|
| Training | {best_row['train_loss']:.4f} | {best_row['train_acc']:.4f} | {best_row.get('train_macro_f1', 0):.4f} |
| Validation | {best_row['val_loss']:.4f} | {best_row['val_acc']:.4f} | {best_row.get('val_macro_f1', 0):.4f} |"""

# Buat section Evaluation Results yang diisi nilai aktual
eval_section_main = f"""### 7.1 Metrik Utama (Validation Set)

| Metrik | Nilai |
|---|---|
| ⭐ **Macro F1 Score** (BDC Primary) | **{eval_metrics['macro_f1']:.4f}** |
| Accuracy | {eval_metrics['accuracy']:.4f} |
| Macro Precision | {eval_metrics['precision']:.4f} |
| Macro Recall | {eval_metrics['recall']:.4f} |"""

cls_rows = '\n'.join([
    f"| {cls} | {eval_metrics['per_class'][cls]['precision']:.4f} | {eval_metrics['per_class'][cls]['recall']:.4f} | {eval_metrics['per_class'][cls]['f1']:.4f} | {eval_metrics['per_class'][cls]['support']} |"
    for cls in CLASS_NAMES.values()
])
macro_row = f"| **Macro Avg** | **{eval_metrics['precision']:.4f}** | **{eval_metrics['recall']:.4f}** | **{eval_metrics['macro_f1']:.4f}** | {sum(eval_metrics['per_class'][c]['support'] for c in CLASS_NAMES.values())} |"

eval_section_cls = f"""### 7.2 Classification Report

| Kelas | Precision | Recall | F1 Score | Support |
|---|---|---|---|---|
{cls_rows}
{macro_row}"""

# Ganti placeholder di laporan
replacements = [
    ('### 6.1 Training Summary\n\n| Metrik | Nilai |\n|---|---|\n| Total Epochs Dijalankan | _(isi setelah training)_ |\n| Best Epoch | _(isi setelah training)_ |\n| Early Stopped | _(Ya/Tidak)_ |\n| Total Training Time | _(menit)_ |',
     train_section),
    
    ('### 7.1 Metrik Utama (Validation Set)\n\n| Metrik | Nilai |\n|---|---|\n| ⭐ **Macro F1 Score** (BDC Primary) | _(isi setelah evaluasi)_ |\n| Accuracy | _(isi)_ |\n| Macro Precision | _(isi)_ |\n| Macro Recall | _(isi)_ |',
     eval_section_main),
    
    ('**Kelas terkuat:** _(isi)_ — F1 = _(isi)_\n**Kelas terlemah:** _(isi)_ — F1 = _(isi)_',
     f'**Kelas terkuat:** {best_cls} — F1 = {f1_vals[best_cls]:.4f}\n**Kelas terlemah:** {worst_cls} — F1 = {f1_vals[worst_cls]:.4f}'),
]

for old_text, new_text in replacements:
    if old_text in report_text:
        report_text = report_text.replace(old_text, new_text)
        print(f'✅ Replaced: {old_text[:50]}...')
    else:
        print(f'⚠️  Placeholder tidak ditemukan (mungkin sudah diupdate): {old_text[:50]}...')

# Tulis kembali laporan
REPORT_PATH.write_text(report_text, encoding='utf-8')
print(f'\n✅ Laporan berhasil diperbarui: {REPORT_PATH}')